Basic Components
====

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tiktoken

### 1. Download data

In [2]:

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


url = "https://www.gutenberg.org/ebooks/2701.txt.utf-8"
repo_root = find_repo_root()
data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

text_path = data_dir / "moby_dick.txt"

if not text_path.exists():
    urlretrieve(url, text_path)

text = text_path.read_text(encoding="utf-8")

print(f"Saved to: {text_path}")
print(f"Characters: {len(text):,}")
print(text[:500])


Saved to: /home/pyml/git_vmirly/llm-recipes/data/raw/moby_dick.txt
Characters: 1,238,279
The Project Gutenberg eBook of Moby Dick; Or, The Whale
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBo


### 2. Tokenization

In [6]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text)

n_vocab = tokenizer.n_vocab

# number of unique tokens in the tokenizer's vocabulary
print(f"Number of unique tokens: {n_vocab}")

print(f"Tokens: {len(token_ids):,}")
print(f"First 10 token IDs: {token_ids[:10]}")

# decoding:
decoded_text = tokenizer.decode(token_ids[:10])
print(f"Decoded text: {decoded_text}")


Number of unique tokens: 50257
Tokens: 331,691
First 10 token IDs: [464, 4935, 20336, 46566, 286, 16540, 88, 11740, 26, 1471]
Decoded text: The Project Gutenberg eBook of Moby Dick; Or


### 3. Embeddings

In [17]:
embedding_dim = 768
context_length = 1024

In [ ]:
import torch
import torch.nn as nn

token_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

print(f"Token embeddings shape: {token_embeddings.weight.shape}")

# testing:
input_ids = torch.tensor(token_ids[:10])
tok_emb = token_embeddings(input_ids)

print(f"input_ids.shape: {input_ids.shape}")   # [10]
print(f"tok_emb.shape: {tok_emb.shape}")     # [10, 768]

tok_emb

Token embeddings shape: torch.Size([50257, 768])
input_ids.shape: torch.Size([10])
tok_emb.shape: torch.Size([10, 768])


tensor([[-0.8934,  0.2257,  0.5436,  ..., -0.1146,  0.4273, -0.8015],
        [ 0.0252, -2.0044,  1.5703,  ..., -1.3576, -0.2511,  0.3243],
        [ 1.5797, -0.2687, -0.8738,  ..., -1.5569, -0.8753, -0.8661],
        ...,
        [-0.5654, -0.0530,  0.4023,  ..., -0.7160,  0.6512,  0.9196],
        [ 0.5612,  1.5293, -1.5703,  ...,  1.4968, -0.9339,  0.6168],
        [ 0.8592, -1.2191,  0.8000,  ..., -0.9812, -0.0229,  0.3840]],
       grad_fn=<EmbeddingBackward0>)

In [19]:
pos_embeddings = nn.Embedding(
    num_embeddings=context_length, embedding_dim=embedding_dim
)

seq_len = input_ids.shape[0]

positions = torch.arange(seq_len)
pos_emb = pos_embeddings(positions)
print(f"pos_emb.shape: {pos_emb.shape}")     # [10, 768]

pos_emb

pos_emb.shape: torch.Size([10, 768])


tensor([[-1.4126,  0.2987, -0.9261,  ...,  1.8204, -0.1191,  0.2857],
        [ 1.7986, -0.1610,  0.8493,  ..., -0.6079,  0.6071,  0.3596],
        [ 1.6924, -0.3941,  0.3997,  ..., -1.7752, -1.1250,  1.0340],
        ...,
        [-0.9699,  1.2890, -2.2598,  ...,  0.3828, -1.2246, -0.3212],
        [-0.5871, -1.4303, -0.4442,  ..., -0.3926,  0.2826, -0.6857],
        [ 1.2967, -1.7834,  3.3246,  ..., -0.1168, -0.9101,  0.9098]],
       grad_fn=<EmbeddingBackward0>)

In [20]:
# adding token embeddings and positional embeddings

x = tok_emb + pos_emb
print(f"x.shape: {x.shape}")  # [10, 768]

x.shape: torch.Size([10, 768])


## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py